# 01 — Data analysis and preparation

This notebook loads the public English Amazon review corpus, documents the star-to-sentiment policy, checks data quality, visualizes labels/review lengths, and creates product-grouped partitions.

In [ ]:
from src.data import load_amazon_reviews, group_train_val_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 20k rows is a quick laptop-friendly first run; increase to 60k or None for a larger study.
df = load_amazon_reviews(max_rows=20000)
df.head(), df.shape

In [ ]:
print(df.info())
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicate text rows:', df.duplicated('text').sum())
print('\nLabel counts:\n', df.label.value_counts())

In [ ]:
plt.figure(figsize=(7,4)); sns.countplot(data=df, x='label', order=['Negative','Neutral','Positive']); plt.title('Sentiment label distribution'); plt.show()
df['review_length'] = df.text.str.split().str.len()
plt.figure(figsize=(8,4)); sns.histplot(df.review_length.clip(upper=df.review_length.quantile(.99)), bins=40); plt.title('Review length (words; clipped at 99th percentile)'); plt.show()

In [ ]:
print(df.groupby('label').review_length.describe())
print('\nTop product categories:\n', df.product_category.value_counts().head(15))

In [ ]:
train, val, test = group_train_val_test_split(df)
for name, part in [('train',train),('validation',val),('test',test)]:
    print(name, len(part), part.label.value_counts(normalize=True).round(3).to_dict())
print('Product overlap train/test:', len(set(train.product_id)&set(test.product_id)))

# Save notebook-sized sample partitions for experiments.
from src.config import DATA_PROCESSED
for name, part in [('train',train),('validation',val),('test',test)]: part.to_csv(DATA_PROCESSED/f'{name}.csv',index=False)

## Interpretation prompts

- Is the class distribution balanced after mapping stars to sentiment?
- Are there empty/duplicate reviews or unusual lengths?
- Why can product-group splitting be more realistic than random row splitting?
- Explain that star ratings are weak labels and can disagree with the wording of mixed reviews.